# 归档材料

保留此材料用于局部机制学习；先修、结论与下游连接需要结合归档索引审查。

[归档索引](../README.md) · [当前学习入口](../../../course/first_loop/README.md)

# 09 · Safety & Runtime：accuracy 之外的发布门禁

本章把旧版 `04`、`18` 和 `15` 合并。一个 learned policy 可能置信度高但输入 stale、定位漂移、TTC 很小或 p99 超时；L4 系统需要独立的 health/safety state machine 和 degraded mode，并把 runtime 证据加入 gate。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

eval_report = load_json_artifact("08_eval_report.json")
model_metrics = load_json_artifact("05_bev_metrics.json") if (ARTIFACT_DIR / "05_bev_metrics.json").exists() else {"runtime": {}}

In [ ]:
def safety_state(sensor_health, localization_sigma_m, latency_ms, ttc_s, odd_ok=True):
    if not odd_ok:
        return "ODD_EXIT"
    if sensor_health < 0.45 or localization_sigma_m > 1.5 or latency_ms > 150 or ttc_s < 1.0:
        return "MINIMAL_RISK"
    if sensor_health < 0.70 or localization_sigma_m > 0.8 or latency_ms > 100 or ttc_s < 2.0:
        return "DEGRADED"
    return "NOMINAL"

test_rows = []
for sensor_health in [0.95, 0.62, 0.35]:
    for ttc in [3.5, 1.6, 0.7]:
        test_rows.append({"sensor_health": sensor_health, "ttc_s": ttc,
                          "state": safety_state(sensor_health, 0.4, 85, ttc)})
state_table = pd.DataFrame(test_rows)
display(state_table)
print("state counts:", state_table.state.value_counts().to_dict())

## Runtime evidence: p50/p95/p99 and watchdog violations

Runtime is not an afterthought. Report batch size, hardware, warm-up policy, precision, model version and latency distribution. A p50 that passes while p99 violates the watchdog still produces a system failure.

In [ ]:
rng = np.random.default_rng(51)
latency = 60.0 + rng.lognormal(mean=2.0, sigma=0.32, size=2000)
runtime_table = {
    "p50_ms": float(np.percentile(latency, 50)),
    "p95_ms": float(np.percentile(latency, 95)),
    "p99_ms": float(np.percentile(latency, 99)),
    "watchdog_budget_ms": 100.0,
    "watchdog_violation_rate": float(np.mean(latency > 100.0)),
}
runtime_table.update({f"model_{key}": value for key, value in model_metrics.get("runtime", {}).items()})
print(runtime_table)
plt.hist(latency, bins=40, alpha=0.8)
plt.axvline(100, color="red", linestyle="--", label="watchdog")
plt.legend()
plt.xlabel("latency / ms")
plt.title("Runtime distributions, not just averages")
plt.show()

In [ ]:
def release_gate(collision_rate, fallback_recall, p95_ms, p99_ms, max_collision=0.05):
    checks = {
        "collision_rate": collision_rate < max_collision,
        "fallback_recall": fallback_recall >= 0.95,
        "p95_latency": p95_ms < 100.0,
        "p99_latency": p99_ms < 130.0,
    }
    return checks, all(checks.values())

checks, passed = release_gate(eval_report["collision_rate"], 0.97,
                              runtime_table["p95_ms"], runtime_table["p99_ms"])
print("release gate:", checks, "PASS" if passed else "HOLD / INVESTIGATE")
save_json_artifact("09_safety_runtime.json", {
    "state_table": state_table.to_dict(orient="records"),
    "runtime": runtime_table,
    "release_checks": checks,
    "next": "10_capstone.ipynb",
})

### 完成标准

构造至少三个 adversarial cases：高 confidence + stale input、低 TTC + nominal latency、p99 超时 + 看似高 accuracy。每个 case 都要有检测信号、系统状态、fallback/degraded action 和恢复条件。不要把 safety gate 当成“把模型阈值调低”。